In [22]:
import math
import time
import numpy as np
import pandas as pd
import yfinance as yf


import torch
import torch.nn as nn

from model import train_pinn


from montecarlo import mc_asian_call_arith
from montecarlo import pinn_price_real


from evaluation import evaluate_mae_pinn_vs_mc
from evaluation import parity_plot_plotly
#from evaluation import run_full_evaluation


SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float32


## WTI Domain

In [23]:
# Get real WTI data and infer ranges

df = yf.download("CL=F", start="2010-01-01", progress=False)
prices = df["Close"].dropna()


S0 = float(prices.iloc[-1])
S_max_hist = float(prices.max())


log_ret = np.log(prices / prices.shift(1)).dropna()
sigma_hist = float(log_ret.std()) * np.sqrt(252)


T = 1.0
S_max_real = 1.2 * S_max_hist
I_max_real = S_max_real * T
sigma_max = min(1.0, 2.0 * sigma_hist)
r_max = 0.5


print("=== REAL (USD) ranges ===")
print(f"S0        = {S0:.2f}")
print(f"S_max     = {S_max_real:.2f}")
print(f"I_max     = {I_max_real:.2f}")
print(f"sigma     = {sigma_hist:.3f}")
print(f"sigma_max = {sigma_max:.3f}")
print(f"r_max     = {r_max}")


=== REAL (USD) ranges ===
S0        = 57.32
S_max     = 148.44
I_max     = 148.44
sigma     = 0.404
sigma_max = 0.807
r_max     = 0.5


/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_74546/968340630.py:7: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_74546/968340630.py:8: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead

/opt/homebrew/lib/python3.11/site-packages/pandas/core/internals/blocks.py:395: RuntimeWarning:

invalid value encountered in log

/var/folders/7y/h7b4hxf568n1y6xlpl8vf6340000gn/T/ipykernel_74546/968340630.py:12: FutureWarning:

Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead



In [3]:
# Normalization: K_scale = S0  (so S̃0 = 1)

K_scale = S0  


S_max = S_max_real / K_scale
I_max = I_max_real / K_scale


print("\n=== NORMALIZED (dimensionless) ranges ===")
print(f"K_scale   = {K_scale:.2f}")
print(f"S0_tilde  = {S0 / K_scale:.3f}")      # should be 1.000
print(f"S_max_t   = {S_max:.3f}")
print(f"I_max_t   = {I_max:.3f}")
print(f"r_max     = {r_max:.3f}")
print(f"sigma_max = {sigma_max:.3f}")



=== NORMALIZED (dimensionless) ranges ===
K_scale   = 57.41
S0_tilde  = 1.000
S_max_t   = 2.586
I_max_t   = 2.586
r_max     = 0.500
sigma_max = 0.807


## Training

In [4]:
model = train_pinn(
   S_max, I_max, r_max, sigma_max,
   width=160,
   depth=4,
   n_epochs=50000,     # start here; increase toward 200k/500k if needed
   lr0=1e-3,
   Np=1000,
   n_bc_axis=100,
   w_pde=1.0,
   print_every=2000
)

K_real = S0  # ATM
r_eval = 0.05
sigma_eval = sigma_hist



print("Evaluation (real scale)")
test_S = [0.8*S0, 1.0*S0, 1.2*S0]  # around spot


for S_test in test_S:
   pinn_p = pinn_price_real(model, S_test, K_real, r_eval, sigma_eval, t0=0.0)
   mc_p, mc_se = mc_asian_call_arith(S_test, K_real, r_eval, sigma_eval, T=1.0, n_steps=252, n_paths=50_000, antithetic=True)


   print(f"S={S_test:8.2f} | PINN={pinn_p:10.4f} | MC={mc_p:10.4f} ± {1.645*mc_se:8.4f} (90% CI half-width)")


ep=   2000 | loss=6.603e-04 | pde=2.435e-04 | data=4.168e-04 | lr=9.96e-04 | 1.2 min
ep=   4000 | loss=2.178e-04 | pde=1.042e-04 | data=1.136e-04 | lr=9.84e-04 | 2.4 min
ep=   6000 | loss=1.044e-04 | pde=4.859e-05 | data=5.579e-05 | lr=9.65e-04 | 3.6 min
ep=   8000 | loss=2.519e-04 | pde=2.699e-05 | data=2.249e-04 | lr=9.38e-04 | 4.9 min
ep=  10000 | loss=8.010e-05 | pde=3.304e-05 | data=4.706e-05 | lr=9.05e-04 | 6.4 min
ep=  12000 | loss=3.276e-05 | pde=1.615e-05 | data=1.661e-05 | lr=8.64e-04 | 7.8 min
ep=  14000 | loss=2.592e-05 | pde=1.359e-05 | data=1.234e-05 | lr=8.19e-04 | 9.2 min
ep=  16000 | loss=1.113e-04 | pde=8.725e-06 | data=1.026e-04 | lr=7.68e-04 | 10.4 min
ep=  18000 | loss=2.109e-05 | pde=6.740e-06 | data=1.435e-05 | lr=7.13e-04 | 11.7 min
ep=  20000 | loss=2.118e-05 | pde=8.697e-06 | data=1.248e-05 | lr=6.55e-04 | 13.0 min
ep=  22000 | loss=3.167e-05 | pde=2.435e-05 | data=7.319e-06 | lr=5.94e-04 | 14.3 min
ep=  24000 | loss=4.705e-05 | pde=2.303e-05 | data=2.402e-05 

## Evaluation

In [20]:
S_grid_eval = np.linspace(0, S_max_real, 300)

results = run_full_evaluation(
    model=model,          # PINN già addestrata
    S_grid=S_grid_eval,   # grid su S
    K_real=K_real,        # strike reale
    r=r_eval,             # risk-free rate
    sigma=sigma_eval,     # volatilità
    T=1.0,                # maturità
    n_steps_mc=252,       # discretizzazione MC
    n_paths_mc=200_000,   # paths MC
    h_delta=1.0,          # bump per delta
)

Running Monte Carlo reference (prices + delta, single pass)...
MAE (PINN vs MC): 0.019105
Speed-up (MC / PINN): 19573.3x


In [5]:
S_grid_real = np.linspace(0.0, S_max_real, 250)


mae, mc_vals, pinn_vals = evaluate_mae_pinn_vs_mc(
   model=model,
   S_grid_real=S_grid_real,
   K_real=K_real,
   r=r_eval,
   sigma=sigma_eval,
   T=1.0,
   n_steps_mc=252,
   n_paths_mc=100_000
)

print(f"MAE (PINN vs MC) = {mae:.6f}")

parity_plot_plotly(
   mc_vals,
   pinn_vals,
   title="Asian Call (WTI) – PINN vs Monte Carlo"
)


Running MC reference pricing...
Evaluating PINN prices...
MAE (PINN vs MC) = 0.022036
